# Treinamento de Modelos: MLflow
## O "Dataset" Iris com MLflow "Autolog"

Antes de usarmos ferramentas de automação, precisamos entender como o **MLflow** intercepta nosso treinamento tradicional para garantir a reprodutibilidade. O Unity Catalog agora atua nativamente como o nosso **Model Registry** (Repositório de Modelos).

In [0]:
%%capture
%pip install databricks
%pip install databricks-automl --upgrade
dbutils.library.restartPython()

## Vamos recriar a tabela do dataset iris com a coluna `id_flor`

In [0]:
# Catalogo e Esquema do dataset Iris
catalogo_origem = "workspace"
esquema_origem = "default"
tabela_iris_raw = f"{catalogo_origem}.{esquema_origem}.iris_dataset"

# Lendo os dados brutos que já estão no ambiente
df_iris = spark.table(tabela_iris_raw)

# Garanta que a tabela possua a chave primária (id_flor) e as colunas ajustadas.

# Registra o DataFrame atual como uma View temporária no Spark
df_iris.createOrReplaceTempView("vw_iris_raw")

# Executa a query SQL para construir a chave primária
df_iris_sql = spark.sql("""
    SELECT 
        CONCAT('flor_', ROW_NUMBER() OVER (ORDER BY (SELECT NULL))) AS id_flor,
        sepal_length,
        sepal_width,
        petal_length,
        petal_width,
        species 
    FROM vw_iris_raw
""")

In [0]:
df_iris_sql.display()

In [0]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from databricks import feature_engineering
from databricks.feature_engineering import FeatureLookup
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Inicializando o cliente da Feature Store e montando o Training Set
fe = feature_engineering.FeatureEngineeringClient()
df_target = df_iris_sql.select("id_flor", "species")

tabela_feature = "workspace.default.iris_features_table"



In [0]:
df_target.display()

### FeatureLookup e create_training_set

Para treinar um modelo utilizando dados da nossa Feature Store, nós não fazemos `JOINs` manuais com o Spark SQL. Em vez disso, deixamos que o `FeatureEngineeringClient` monte a matriz de treino de forma inteligente e segura através de duas etapas:

#### 1. `FeatureLookup`
O `FeatureLookup` funciona como uma "lista de compras". Ele avisa ao Databricks quais variáveis específicas você quer resgatar, de qual tabela do Unity Catalog elas vêm, e qual chave deve ser usada para o cruzamento.

* **`table_name`**: O caminho completo da sua Feature Table no Unity Catalog.
* **`feature_names`**: Uma lista com os nomes exatos das colunas (recursos) que você quer trazer para o modelo.
* **`lookup_key`**: A chave primária de amarração (ex: `id_flor` ou `id_usuario`).
* **`timestamp_lookup_key`** *(Opcional para Séries Temporais)*: A coluna que define o momento do corte temporal, garantindo a consistência histórica ("Point-in-time correctness").

#### `fe.create_training_set`
Este método é o motor que junta a tabela que contém a variável resposta com as definições do `FeatureLookup`.

* **`df`**: O seu DataFrame base contendo as chaves primárias e a label de treino.
* **`feature_lookups`**: A lista de objetos `FeatureLookup` que definimos no passo anterior.
* **`label`**: O nome da coluna alvo que o modelo tentará prever.
* **`exclude_columns`**: Lista de colunas que devem ser removidas do DataFrame final de treino (como IDs ou carimbos de data/hora), evitando que o algoritmo use dados de identificação para tomar decisões.


In [0]:

features_lookup = [
    FeatureLookup(
        table_name=tabela_feature,
        feature_names=["sepal_length", "sepal_width", "petal_length", "petal_width"],
        lookup_key=["id_flor"],
    )
]

training_set = fe.create_training_set(
    df=df_target,
    feature_lookups=features_lookup,
    label="species",
    exclude_columns=["id_flor"],
)


In [0]:
# Convertendo para Pandas para o ecossistema Scikit-Learn
df_train = training_set.load_df().toPandas()
X = df_train.drop(columns=["species"])
y_raw = df_train["species"]
# Convertendo os alvos de texto para inteiros (0, 1, 2)
le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### fe.log_model

O método `fe.log_model` é o passo definitivo de governança no ecossistema de "Machine Learning" do Databricks. Em vez de salvar o modelo de forma isolada como um binário comum (`mlflow.sklearn.log_model`), nós utilizamos o cliente da Feature Store para envelopar o modelo junto com os seus **metadados de origem**.

* **`model`**: A instância do modelo matemático treinado no Scikit-Learn, XGBoost, etc.
* **`artifact_path`**: O nome da pasta interna onde os arquivos do modelo (binários, dependências) serão salvos no MLflow.
* **`flavor`**: O "sabor" do framework do modelo (ex: `mlflow.sklearn`, `mlflow.xgboost`).
* **`training_set`**: O objeto gerado pelo `fe.create_training_set`. É daqui que o método extrai a receita de quais chaves e variáveis pertencem ao modelo.
* **`registered_model_name`**: O caminho de três níveis no Unity Catalog (`catalogo.esquema.nome_do_modelo`) para catalogar e versionar o modelo oficialmente.

In [0]:
# Autolog Global
mlflow.sklearn.autolog(log_models=False)

# Treinando vários modelos
modelos_para_testar = {
    "Logistic_Regression": LogisticRegression(),
    "Decision_Tree": DecisionTreeClassifier(),
    "Random_Forest": RandomForestClassifier(),
    "Gradient_Boosting": GradientBoostingClassifier()
}

for nome_modelo, instancia_modelo in modelos_para_testar.items():
    
    with mlflow.start_run(run_name=f"Model_{nome_modelo}") as run:
        print(f"Treinando e avaliando: {nome_modelo}...")
    
        # O Autolog intercepta o treinamento e salva os parâmetros de cada arquitetura
        instancia_modelo.fit(X_train, y_train)

        # Calculando e registrando a acurácia no conjunto de teste
        acuracia_teste = instancia_modelo.score(X_test,y_test)
        mlflow.log_metric("acuracia_teste", acuracia_teste)

        print(f"Acurácia no conjunto de teste: {acuracia_teste}")
        
        # Registrando o modelo envelopado com os metadados da Feature Store no Unity Catalog
        fe.log_model(
            model=instancia_modelo,
            artifact_path=f"model_{nome_modelo.lower()}",
            flavor=mlflow.sklearn,
            training_set=training_set,
            registered_model_name= f"{catalogo_origem}.{esquema_origem}.iris_model_prod"
        )


####  Por que usar o `fe.log_model` em vez do log padrão do MLflow?

Quando você registra um modelo através deste método, o Unity Catalog cria um vínculo indestrutível entre o modelo treinado e as tabelas de recursos. Isso traz duas vantagens críticas para a produção:

1. **Linhagem de Dados Automatizada ("Lineage"):** No painel do Unity Catalog, você conseguirá ver exatamente quais tabelas e colunas da Feature Store alimentam esse modelo. Se uma feature mudar na origem, você sabe exatamente quais modelos serão afetados.
2. **Auto-Lookup em Produção (Model Serving):** Quando esse modelo for implantado como uma API em tempo real, a aplicação cliente precisa enviar apenas a chave primária (ex: `id_usuario`). O endpoint do modelo consulta a Feature Store de baixíssima latência ("Online Tables") de forma totalmente transparente e automática para buscar o valor atualizado das variáveis, eliminando a necessidade de o desenvolvedor de software codificar cruzamentos complexos na API.

## O que é o Databricks AutoML?

O **Databricks AutoML** é um assistente de "Machine Learning". Diferente de outras ferramentas de mercado que geram um arquivo binário fechado e inacessível, o AutoML do Databricks executa o pipeline de modelagem de forma transparente.

Para cada modelo testado, o AutoML gera:
1. **Um Notebook Python Completo:** Contendo o código exato de engenharia de dados, seleção de variáveis e hiperparâmetros utilizados.
2. **Reprodutibilidade:** O cientista de dados pode clonar o código gerado, customizar a arquitetura e estender o modelo manualmente.

## Modelos e Algoritmos Suportados

O ecossistema clássico do Databricks AutoML está dividido em três grandes pilares de problemas preditivos, utilizando as bibliotecas mais robustas do mercado:

| Tipo de Problema | Algoritmos Avaliados | Bibliotecas Base |
| :--- | :--- | :--- |
| **Classificação** | *XGBoost*, *LightGBM*, *Random Forest*, *Decision Trees*, *Logistic Regression* | `scikit-learn`, `xgboost`, `lightgbm` |
| **Regressão** | *XGBoost*, *LightGBM*, *Random Forest*, *Linear Regression* con regularização | `scikit-learn`, `xgboost`, `lightgbm` |
| **Previsão Temporal** (*Forecasting*) | *Prophet*, *Auto-ARIMA* | `prophet`, `statsmodels` |

## Foco em "Forecasting"

Na versão gratuíta temos apenas a infraestrutura **Serverless** do Databricks. 

**Restrição Técnica do Ambiente:** Por questões de arquitetura, a funcionalidade do AutoML baseada em interface gráfica ("UI") está otimizada e homologada especificamente para tarefas de **"Time-Series Forecasting" (Previsão de Séries Temporais)**. 

Por esse motivo, nosso projeto prático de automação focará na previsão de tendências temporais (utilizando o *dataset* histórico de COVID-19 como *baseline*).

## Como Executar o AutoML no Ambiente Serverless

Para iniciar o seu primeiro experimento de *Forecasting* sem escrever nenhuma linha de código, siga o roteiro abaixo:

1. No menu lateral esquerdo do Databricks, mude o perfil de desenvolvimento para **"Machine Learning"**.
2. Clique no menu **"Experiments"** (Experimentos) e selecione **"Create AutoML Experiment"**.
3. No campo **"Compute"**, certifique-se de selecionar a opção **Serverless**.
4. Em **"ML problem type"**, altere para **"Forecasting"**.
5. Selecione a tabela de origem guardada no Unity Catalog
6. Configure as variáveis fundamentais da série temporal:
   * **"Prediction target":** A coluna numérica que deseja prever (ex: `cases`).
   * **"Time column":** A coluna que dita o tempo (ex: `date`).
7. Clique em **"Start AutoML"**.

Ao final do processo, explore o "Leaderboard" gerado, clique nos links para inspecionar os notebooks de código gerados pelo Prophet e compare as métricas de erro estatístico (MAE, RMSE) direto na interface gráfica!

Há uma variedade de conjuntos de dados de amostra fornecidos pelo site Databricks e disponibilizados por terceiros que o senhor pode usar no seu Databricks workspace.

Podemos consultar os dataset pelo comando abaixo:

In [0]:
display(dbutils.fs.ls('/databricks-datasets'))

In [0]:
from pyspark.sql.functions import col, to_date

# Configuração dos caminhos do Unity Catalog
catalogo = "workspace"
esquema = "default"
tabela_destino = f"{catalogo}.{esquema}.bike_sharing_forecast"

print("Lendo o CSV de Bike Sharing via Spark DataFrame API...")

# Leitura direta do arquivo DBFS
df_bike_spark = spark.read.csv(
    "dbfs:/databricks-datasets/bikeSharing/data-001/day.csv",
    header=True,
    inferSchema=True
)

# Aplicando as transformações nativas do Spark
df_bike_preparado = df_bike_spark \
    .withColumn("dteday", to_date(col("dteday"))) \
    .withColumn("cnt", col("cnt").cast("int"))

print(f"Salvando diretamente no Unity Catalog na tabela: {tabela_destino}...")

# Escrita direta na camada Delta do catálogo
df_bike_preparado.write.mode("overwrite").saveAsTable(tabela_destino)

print("Concluído! O dataset está pronto para o experimento de Forecasting.")

# Visualizando o resultado final
display(df_bike_preparado.limit(5))